# Section 5 — Results and Economic Interpretation

**ECON 430 Project 1 · 25 points**

[5(a)](#current-5a) · [5(b)](#current-5b) · [5(c)](#current-5c) · [5(d)](#current-5d) · [5(e)](#current-5e)

This is the current submission analysis. Requirements follow the [professor’s PDF](../project1-1%20%281%29.pdf). Section 4 letters organize its unlettered diagnostic bullets. The prior analysis is preserved verbatim in the historical appendix at the end; its old results/statuses do not describe the current model. Current Model 3 uses the explicit full-candidate significance rule in Section 3(d); the former seven-predictor model remains supporting evidence.

**Execution:** fresh Python kernel, working directory `notebooks/`; install the compatible project requirements. Input is the unchanged [4,139-space processed CSV](../data/processed/parking_cross_section.csv), built by [cleaning/03](cleaning/03_build_cross_section.ipynb). Main 01/06 are narrative; 02–05 reload their own inputs. [01](01_economic_theory.ipynb) · [02](02_data_description_eda.ipynb) · [03](03_variable_selection_models.ipynb) · [04](04_diagnostics_robustness.ipynb) · [05](05_results_interpretation.ipynb) · [06](06_limitations_extensions.ipynb).

<a id="current-5a"></a>

## 5(a) Professional Final Regression Table

**Requirement:** Present current final model with clearly labeled uncertainty.

**Method:** Present every coefficient with conventional and HC1 uncertainty, N and fit. **Evidence:** current code/outputs or cited substantive discussion immediately below. **Interpretation:** accompanies the evidence, including units and limitations. **Status:** Complete for analytical coverage; execution evidence is recorded separately, and no grade is guaranteed.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display, Markdown

%matplotlib inline

root = Path.cwd().parent
parking_full = pd.read_csv(root / "data/processed/parking_cross_section.csv",
                          dtype={"SpaceID": str, "GEOID": str})
parking = parking_full.loc[parking_full["RateType"].isin(["FLAT", "TOD"])].copy()
dependent_variable = "occupancy_rate"
candidate_variables = [
    "price_midpoint", "time_limit_hours", "population_density", "median_income",
    "nearby_restaurant_retail_count", "drive_alone_share", "RateType", "MeterType",
    "housing_density", "transit_share", "walk_share", "nearby_business_count"
]
encoded = pd.get_dummies(parking[candidate_variables], drop_first=True, dtype=int)
data = pd.concat([parking[["occupancy_rate"]], encoded], axis=1).dropna().copy()
y = data["occupancy_rate"]
X = data.drop(columns="occupancy_rate")
assert parking_full["SpaceID"].is_unique
assert parking.loc[data.index, "SpaceID"].is_unique
print("Original / FLAT+TOD / common:", len(parking_full), len(parking), len(data))
print("Candidates / common-sample tracts:", X.shape[1], parking.loc[data.index, "GEOID"].nunique())
# Fixed rule: retain EVERY slope with conventional p < 0.05 in the
# original 12-candidate full OLS model, on the same complete-case sample.
# This is one screen, not repeated p-value deletion or hold-out tuning.
full_model = sm.OLS(y, sm.add_constant(X)).fit()
alpha = 0.05
predictors = [v for v in X.columns if full_model.pvalues[v] < alpha]
legacy_predictors = ["price_midpoint", "time_limit_hours", "population_density",
                     "nearby_restaurant_retail_count", "drive_alone_share",
                     "transit_share", "RateType_TOD"]
X1 = sm.add_constant(data[["price_midpoint"]])
X2 = sm.add_constant(data[["price_midpoint", "population_density", "nearby_restaurant_retail_count"]])
X3 = sm.add_constant(data[predictors])
model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
hc1 = sm.OLS(y, X3).fit(cov_type="HC1")
legacy_model = sm.OLS(y, sm.add_constant(data[legacy_predictors])).fit()
assert set(predictors) == set(full_model.pvalues.drop("const").loc[lambda p: p < alpha].index)


Original / FLAT+TOD / common: 4139 4060 4016
Candidates / common-sample tracts: 12 45


In [2]:
regression_table=pd.DataFrame({"OLS coefficient":model3.params,"Conventional SE":model3.bse,
 "HC1 SE":hc1.bse,"Conventional p":model3.pvalues,"HC1 p":hc1.pvalues,
 "HC1 95% lower":hc1.conf_int()[0],"HC1 95% upper":hc1.conf_int()[1]})
regression_table.index.name="Parameter"
display(regression_table.style.format("{:.6g}"))
display(Markdown(f"**Outcome:** monthly occupancy fraction. **N:** {int(model3.nobs):,}; **slopes:** {len(predictors)}; **R²:** {model3.rsquared:.5f}; **adjusted R²:** {model3.rsquared_adj:.5f}. HC1 uses statsmodels' default normal-reference inference; conventional and robust intervals should not be mixed."))


,OLS coefficient,Conventional SE,HC1 SE,Conventional p,HC1 p,HC1 95% lower,HC1 95% upper
Parameter,,,,,,,
const,0.319517,0.00938566,0.0105497,2.35919e-223,1.70823e-201,0.29884,0.340194
price_midpoint,0.0515432,0.00231509,0.00224949,1.24623e-103,3.42795e-116,0.0471343,0.0559521
population_density,3.96115e-06,2.16052e-07,2.27304e-07,3.64525e-72,5.18242e-68,3.51564e-06,4.40665e-06
nearby_restaurant_retail_count,-0.000534366,4.21681e-05,3.8711e-05,4.11596e-36,2.41157e-43,-0.000610238,-0.000458494
transit_share,0.100461,0.0449134,0.0459653,0.0253558,0.0288457,0.0103709,0.190552
walk_share,0.282333,0.0286379,0.0267372,1.13582e-22,4.58909e-26,0.229929,0.334737
nearby_business_count,6.66312e-05,9.27183e-06,8.88582e-06,7.89437e-13,6.45059e-14,4.92153e-05,8.40471e-05
RateType_TOD,0.0435671,0.00705204,0.00809947,7.14119e-10,7.4896e-08,0.0276925,0.0594418


**Outcome:** monthly occupancy fraction. **N:** 4,016; **slopes:** 7; **R²:** 0.29414; **adjusted R²:** 0.29291. HC1 uses statsmodels' default normal-reference inference; conventional and robust intervals should not be mixed.

<a id="current-5b"></a>

## 5(b) Economic Interpretation of All Coefficients

**Requirement:** Interpret every coefficient, including the intercept, in economic units.

**Method:** Rescale every slope to economic units and explain the intercept. **Evidence:** current code/outputs or cited substantive discussion immediately below. **Interpretation:** accompanies the evidence, including units and limitations. **Status:** Complete for analytical coverage; execution evidence is recorded separately, and no grade is guaranteed.

In [3]:
changes=pd.Series({"price_midpoint":1,"population_density":1000,
 "nearby_restaurant_retail_count":100,"transit_share":.1,"walk_share":.1,
 "nearby_business_count":100,"RateType_TOD":1})
labels={"price_midpoint":"$1 higher stated hourly proxy",
 "population_density":"1,000 more people per sq mile",
 "nearby_restaurant_retail_count":"100 more food/retail registrations, total fixed",
 "transit_share":"10-pp higher resident transit share",
 "walk_share":"10-pp higher resident walk share",
 "nearby_business_count":"100 more total registrations, food/retail fixed",
 "RateType_TOD":"TOD versus FLAT"}
effects=pd.DataFrame({"Comparison":[labels[v] for v in predictors],
 "Occupancy difference (pp)":100*hc1.params[predictors]*changes[predictors],
 "HC1 lower (pp)":100*hc1.conf_int().loc[predictors,0]*changes[predictors],
 "HC1 upper (pp)":100*hc1.conf_int().loc[predictors,1]*changes[predictors],
 "HC1 p":hc1.pvalues[predictors]})
display(effects.style.format({c:"{:.4g}" for c in effects if c!="Comparison"}))
for v in predictors:
    effect=effects.loc[v,"Occupancy difference (pp)"]
    lo,hi=effects.loc[v,["HC1 lower (pp)","HC1 upper (pp)"]]
    display(Markdown(f"- **{labels[v]}:** associated with **{abs(effect):.3f} percentage points {'higher' if effect>=0 else 'lower'} occupancy**, holding other included covariates fixed; HC1 95% CI **[{lo:.3f}, {hi:.3f}] pp**."))
display(Markdown(f"**Intercept:** {model3.params['const']:.5f} occupancy, or {100*model3.params['const']:.2f}%, for a FLAT space with zero values for all numeric covariates. This joint zero-covariate profile is outside observed support, so it is an algebraic baseline, not an estimate for a typical parking space."))


,Comparison,Occupancy difference (pp),HC1 lower (pp),HC1 upper (pp),HC1 p
price_midpoint,$1 higher stated hourly proxy,5.154,4.713,5.595,3.428e-116
population_density,"1,000 more people per sq mile",0.3961,0.3516,0.4407,5.182e-68
nearby_restaurant_retail_count,"100 more food/retail registrations, total fixed",-5.344,-6.102,-4.585,2.412e-43
transit_share,10-pp higher resident transit share,1.005,0.1037,1.906,0.02885
walk_share,10-pp higher resident walk share,2.823,2.299,3.347,4.589e-26
nearby_business_count,"100 more total registrations, food/retail fixed",0.6663,0.4922,0.8405,6.451e-14
RateType_TOD,TOD versus FLAT,4.357,2.769,5.944,7.49e-08


- **\$1 higher stated hourly proxy:** associated with **5.154 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[4.713, 5.595] pp**.

- **1,000 more people per sq mile:** associated with **0.396 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.352, 0.441] pp**.

- **100 more food/retail registrations, total fixed:** associated with **5.344 percentage points lower occupancy**, holding other included covariates fixed; HC1 95% CI **[-6.102, -4.585] pp**.

- **10-pp higher resident transit share:** associated with **1.005 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.104, 1.906] pp**.

- **10-pp higher resident walk share:** associated with **2.823 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[2.299, 3.347] pp**.

- **100 more total registrations, food/retail fixed:** associated with **0.666 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[0.492, 0.840] pp**.

- **TOD versus FLAT:** associated with **4.357 percentage points higher occupancy**, holding other included covariates fixed; HC1 95% CI **[2.769, 5.944] pp**.

**Intercept:** 0.31952 occupancy, or 31.95%, for a FLAT space with zero values for all numeric covariates. This joint zero-covariate profile is outside observed support, so it is an algebraic baseline, not an estimate for a typical parking space.

Both business counts are in this specification. Holding the total fixed, a higher food/retail count represents a change in registration composition; holding food/retail fixed, a higher total represents other registrations. The negative conditional food/retail coefficient is therefore not the effect of opening new businesses. Shares describe tract residents, and a 10-percentage-point share change is 0.10 in fraction units. No coefficient is interpreted as a proven causal effect.

<a id="current-5c"></a>

## 5(c) Statistical vs. Economic Significance

**Requirement:** Separate precision from economic importance.

**Method:** Compare computed p-values with economically scaled magnitudes. **Evidence:** current code/outputs or cited substantive discussion immediately below. **Interpretation:** accompanies the evidence, including units and limitations. **Status:** Complete for analytical coverage; execution evidence is recorded separately, and no grade is guaranteed.

In [4]:
display(pd.DataFrame({"HC1 p":hc1.pvalues[predictors],"Significant at 5%":hc1.pvalues[predictors]<.05,
                      "Scaled occupancy pp":effects["Occupancy difference (pp)"]}))
display(Markdown(f"A $1 price comparison is {effects.loc['price_midpoint','Occupancy difference (pp)']:.2f} occupancy pp, against a common-sample mean of {100*y.mean():.2f}%. Economic relevance depends on the realistic price change and local support, not just the p-value. Statistical significance is conditional on selection and the covariance assumptions and does not establish causality."))


,HC1 p,Significant at 5%,Scaled occupancy pp
price_midpoint,3.427946e-116,True,5.154319
population_density,5.182421e-68,True,0.396115
nearby_restaurant_retail_count,2.411570e-43,True,-5.343657
transit_share,2.884573e-02,True,1.004612
walk_share,4.589094e-26,True,2.823330
nearby_business_count,6.450594e-14,True,0.666312
RateType_TOD,7.489605e-08,True,4.356712


A \$1 price comparison is 5.15 occupancy pp, against a common-sample mean of 53.76%. Economic relevance depends on the realistic price change and local support, not just the p-value. Statistical significance is conditional on selection and the covariance assumptions and does not establish causality.

<a id="current-5d"></a>

## 5(d) Confidence Intervals for Key Parameters

**Requirement:** Calculate and interpret confidence intervals.

**Method:** Calculate all HC1 intervals and explain coverage and inference limitations. **Evidence:** current code/outputs or cited substantive discussion immediately below. **Interpretation:** accompanies the evidence, including units and limitations. **Status:** Complete for analytical coverage; execution evidence is recorded separately, and no grade is guaranteed.

The tables in 5(a)–5(b) calculate HC1 95% intervals for every parameter and express every slope interval in the same scaled economic units. An interval spanning zero does not establish a nonzero association; an interval excluding zero does not establish an exogenous price effect. The 95% interpretation refers to long-run coverage under the fitted inference assumptions, not a 95% probability attached to a fixed unknown coefficient. Prior selection and correlated neighborhood errors can invalidate nominal coverage; Section 4(e) adds conditional row-bootstrap intervals without claiming to solve those issues.

<a id="current-5e"></a>

## 5(e) R² and Model Fit in Economic Context

**Requirement:** Explain fit, predictive error and literature comparison.

**Method:** Interpret fit/error, compare findings with both studies and state the conclusion. **Evidence:** current code/outputs or cited substantive discussion immediately below. **Interpretation:** accompanies the evidence, including units and limitations. **Status:** Complete for analytical coverage; execution evidence is recorded separately, and no grade is guaranteed.

In [5]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
folds = KFold(n_splits=5, shuffle=True, random_state=10)
fits = {"Model 1": model1, "Model 2": model2, "Model 3 (rubric)": model3,
        "Prior seven (supporting)": legacy_model}
fold_errors = {}
for label, fit in fits.items():
    columns = [v for v in fit.model.exog_names if v != "const"]
    fold_errors[label] = -cross_val_score(LinearRegression(), data[columns], y,
                                        cv=folds, scoring="neg_root_mean_squared_error")
cv_results = pd.DataFrame(fold_errors, index=pd.Index(range(1, 6), name="Held-out fold"))
comparison = pd.DataFrame({
    label: {"Slopes": len(fit.params)-1, "N": int(fit.nobs), "R2": fit.rsquared,
            "Adjusted R2": fit.rsquared_adj, "AIC": fit.aic, "BIC": fit.bic,
            "In-sample RMSE": np.sqrt(np.mean(fit.resid**2)),
            "Mean 5-fold RMSE": fold_errors[label].mean(),
            "SD across folds": fold_errors[label].std(ddof=1)}
    for label, fit in fits.items()
}).T
display(cv_results.round(6))
display(comparison.round(6))
display(Markdown(f"Current Model 3 explains **{100*model3.rsquared:.2f}%** of sample occupancy variation; **{100*(1-model3.rsquared):.2f}%** remains unexplained. Its mean five-fold RMSE is **{100*fold_errors['Model 3 (rubric)'].mean():.2f} occupancy pp**. This monthly cross-sectional error is not an instantaneous availability probability or out-of-city validation."))


,Model 1,Model 2,Model 3 (rubric),Prior seven (supporting)
Held-out fold,,,,
1,0.175393,0.165986,0.162474,0.162926
2,0.179741,0.170995,0.163980,0.166667
3,0.179111,0.170807,0.163985,0.167118
4,0.176845,0.165379,0.162830,0.164252
5,0.176728,0.167872,0.160709,0.165612


,Slopes,N,R2,Adjusted R2,AIC,BIC,In-sample RMSE,Mean 5-fold RMSE,SD across folds
Model 1,1.0,4016.0,0.156775,0.156565,-2483.670513,-2471.074430,0.177522,0.177564,0.001807
Model 2,3.0,4016.0,0.243498,0.242932,-2915.517259,-2890.325092,0.168146,0.168208,0.002626
Model 3 (rubric),7.0,4016.0,0.294138,0.292905,-3185.769994,-3135.385661,0.162421,0.162796,0.001349
Prior seven (supporting),7.0,4016.0,0.273623,0.272354,-3070.712410,-3020.328077,0.164764,0.165315,0.001732


Current Model 3 explains **29.41%** of sample occupancy variation; **70.59%** remains unexplained. Its mean five-fold RMSE is **16.28 occupancy pp**. This monthly cross-sectional error is not an instantaneous availability probability or out-of-city validation.

### Comparison with academic literature

Shoup’s (2006) search-cost mechanism concerns the curb/off-street price tradeoff and scarce spaces. Our positive conditional price association need not contradict it: a cross-location regression can combine strong demand with officials’ higher tariffs, and our outcome does not measure search time. [Shoup (2006)](https://escholarship.org/content/qt55s7079f/qt55s7079f.pdf).

Pierce and Shoup (2013) report an average elasticity about −0.4 from SFpark adjustments, whereas our level coefficient is an occupancy-fraction difference per dollar across LA spaces. The units and designs differ. A positive LA association cannot be interpreted as an opposite causal elasticity; source dates, omitted demand and tariff assignment differ. [Pierce and Shoup (2013)](https://www.its.ucla.edu/wp-content/uploads/sites/10/2017/06/Getting-the-Prices-Right-An-Evaluation-of-Pricing-Parking-by-Demand-in-San-Francisco.pdf).

The conditional commercial-composition estimates and resident mode shares describe spatial context, not identified demand shifts. Our data therefore support historically aligned tariff-change research rather than a claim that higher tariffs cause occupancy to rise. The 1(d) elasticity translation is an illustrative benchmark, not a fitted restriction.

### Conclusion

We retain the negative demand hypothesis as theory, report the observed conditional associations without sign manipulation, and carry the explicitly defined full-candidate-significant Model 3 through actual fit, validation and diagnostic procedures. Economic size, uncertainty and unresolved specification/dependence limits must be read together. Prior seven-predictor estimates remain supporting historical evidence, not mislabeled current results.

In [6]:
table_dir=root/"reports/tables";table_dir.mkdir(parents=True,exist_ok=True)
(table_dir/"current_final_results.json").write_text(json.dumps({
 "n":len(data),"predictors":predictors,"selection_rule":"all conventional p<.05 slopes in original 12-candidate full OLS",
 "coefficients":json.loads(regression_table.to_json(orient="index")),
 "economic_effects":json.loads(effects.to_json(orient="index")),
 "model_comparison":json.loads(comparison.to_json(orient="index"))},indent=2))


5545

## Section Summary and Rubric Coverage

5(a), 5(b), 5(c), 5(d), 5(e) have current methods/evidence or substantive discussion. Computational completion is accepted only after clean-kernel execution passes; names and syllabus review are deferred to final packaging by the group. No grade or causal finding is guaranteed. The historical material below is preserved supporting work, not current status evidence.

# Historical analysis preserved before rubric completion

Everything below is retained verbatim from commit `96d4bb1`. Its old notebook numbers, model definitions, diagnostic conclusions and compliance claims are historical. Use the current analysis above for submission results. No original calculation, explanation, figure or output has been removed.

# Section 5 — Results and Economic Interpretation

**Assignment Section 5: 25 points.** This notebook organizes the existing results and economic interpretation material without changing the analysis.

Authority: [professor’s assignment PDF](../project1-1%20%281%29.pdf), pp. 2–4; [exhaustive evidence audit](../docs/rubric_compliance_audit.md). Requirements are paraphrased in each subsection. Section 4 letters are organizational labels; the PDF uses diagnostic bullets. Historical numbered headings and forward-looking statements inside preserved cells refer to the original eight-notebook workflow; use this roadmap and the [cell migration audit](../docs/notebook_restructure_audit.md) for current locations. Historical compliance claims are preserved as source material; the new evidence audit supersedes them.

### Roadmap

- [5(a)](#rubric-5a)
- [5(b)](#rubric-5b)
- [5(c)](#rubric-5c)
- [5(d)](#rubric-5d)
- [5(e)](#rubric-5e)

### Dependencies and navigation

[Cleaning pipeline](cleaning/03_build_cross_section.ipynb) → [fixed cross-section](../data/processed/parking_cross_section.csv). Run each notebook with its own fresh Python kernel and working directory `notebooks/`. Sections 1 and 6 are narrative; sections 2–5 reload their own inputs. [Theory](01_economic_theory.ipynb) · [EDA](02_data_description_eda.ipynb) · [Models](03_variable_selection_models.ipynb) · [Diagnostics](04_diagnostics_robustness.ipynb) · [Results](05_results_interpretation.ipynb) · [Limitations](06_limitations_extensions.ipynb).

### Reload the unchanged final model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import jarque_bera

%matplotlib inline

parking_all = pd.read_csv("../data/processed/parking_cross_section.csv",
                          dtype={"SpaceID": str, "GEOID": str})
parking = parking_all[parking_all["RateType"].isin(["FLAT", "TOD"])].copy()
parking["RateType_TOD"] = (parking["RateType"] == "TOD").astype(int)
predictors = ["price_midpoint", "time_limit_hours", "population_density",
              "nearby_restaurant_retail_count", "drive_alone_share",
              "transit_share", "RateType_TOD"]
variables = ["occupancy_rate", *predictors, "median_income"]
data = parking[variables].dropna().copy()
y = data["occupancy_rate"]
X3 = sm.add_constant(data[predictors])
model3 = sm.OLS(y, X3).fit()
hc1 = sm.OLS(y, X3).fit(cov_type="HC1")
assert len(data) == 4016
assert parking.loc[data.index, "SpaceID"].nunique() == len(data)
print("Original spaces:", len(parking_all))
print("FLAT + TOD:", len(parking), "Common sample:", len(data))
print("Tracts:", parking.loc[data.index, "GEOID"].nunique())

Original spaces: 4139
FLAT + TOD: 4060 Common sample: 4016
Tracts: 45


The original cross-section has **4,139** spaces. Excluding 79 JUMP meters gives **4,060**; the common comparison excludes 44 missing incomes to match Notebook 05, even though income is not a final regressor. Occupancy is occupied valid minutes divided by all valid minutes, using observed state persistence, excluding UNKNOWN-start intervals and gaps over 24 hours, and requiring at least 25 transaction-active days.

Prices and business registrations are October 2026 snapshots. Census predictors use 2020–2024 ACS five-year estimates, 2024 tract boundaries and Gazetteer land area; business counts use a quarter-mile radius. Notebooks 01–03 and the [source catalog](../docs/) document construction and raw paths; none of those data are rebuilt here.

<a id="rubric-5a"></a>

## 5(a) Professional Final Regression Table

**Requirement:** Present final model results in a professional regression table.

**Method:** Retain the original final table distinguishing conventional and HC1 uncertainty.

**Evidence:** Original 08 cells 13–14.

**Interpretation:** HC1 changes covariance estimates, not coefficients.

**Status: Complete.** 

## 5. Final regression table

The primary model uses all seven preferred predictors and 4,016 spaces. Because both variance tests reject constant variance, I use **HC1** for the final p-values and 95% intervals, following Lecture 2; conventional OLS standard errors and p-values are shown separately for comparison. HC1 leaves coefficients unchanged and still assumes independent observations.

In [7]:
regression_table = pd.DataFrame({
    "OLS coefficient": model3.params,
    "Conventional SE": model3.bse,
    "HC1 SE": hc1.bse,
    "Conventional p-value": model3.pvalues,
    "HC1 p-value": hc1.pvalues,
    "HC1 95% lower": hc1.conf_int()[0],
    "HC1 95% upper": hc1.conf_int()[1]
})
regression_table.index.name = "Variable"
regression_table.style.format({
    "OLS coefficient": "{:.6g}", "Conventional SE": "{:.6g}", "HC1 SE": "{:.6g}",
    "Conventional p-value": "{:.3e}", "HC1 p-value": "{:.3e}",
    "HC1 95% lower": "{:.6g}", "HC1 95% upper": "{:.6g}"})

,OLS coefficient,Conventional SE,HC1 SE,Conventional p-value,HC1 p-value,HC1 95% lower,HC1 95% upper
Variable,,,,,,,
const,0.456337,0.0157939,0.0171155,6.470e-167,1.291e-156,0.422792,0.489883
price_midpoint,0.0556328,0.00236136,0.00232226,4.731e-115,7.931e-127,0.0510813,0.0601844
time_limit_hours,0.00487929,0.00294796,0.00313717,9.797e-02,1.199e-01,-0.00126946,0.011028
population_density,4.57052e-06,2.21771e-07,2.37074e-07,8.803e-90,8.077e-83,4.10586e-06,5.03518e-06
nearby_restaurant_retail_count,-0.000321282,2.29852e-05,2.4298e-05,2.189e-43,6.502e-40,-0.000368905,-0.000273659
drive_alone_share,-0.198101,0.0208314,0.0211007,3.195e-21,6.095e-21,-0.239458,-0.156744
transit_share,0.0196471,0.0468362,0.0452127,6.749e-01,6.639e-01,-0.0689682,0.108262
RateType_TOD,0.0330042,0.00731608,0.00857071,6.631e-06,1.177e-04,0.0162059,0.0498025


<a id="rubric-5b"></a>

## 5(b) Economic Interpretation of All Coefficients

**Requirement:** Interpret every coefficient’s magnitude, direction and units.

**Method:** Retain final HC1 slope scaling and duplicate the original complete coefficient narrative, including the intercept.

**Evidence:** Original 08 cells 15–16 plus original 06 cells 22–23 duplicated below.

**Interpretation:** Occupancy effects are percentage-point differences. The duplicated narrative’s intervals are conventional OLS; use the labeled HC1 table for final inference.

**Status: Complete.** 

## 6. Economic interpretation

A **$1 higher price proxy is associated with about 5.56 percentage points higher occupancy**, holding the included variables constant; the HC1 95% interval is about **5.11 to 6.02 percentage points**. This is a meaningful difference in monthly occupancy, but it is not a causal effect or a price elasticity: the dependent variable is a fraction in levels and the TOD midpoint is a proxy.

Time limit and transit share remain insignificant at 5% under HC1. The table below rescales every slope and its interval into occupancy percentage points, including those uncertain estimates, so statistical precision is kept separate from economic size.

In [8]:
changes = pd.Series({"price_midpoint": 1, "time_limit_hours": 1,
                     "population_density": 1000, "nearby_restaurant_retail_count": 100,
                     "drive_alone_share": 0.10, "transit_share": 0.10, "RateType_TOD": 1})
economic_effects = pd.DataFrame({
    "Comparison": ["$1 higher proxy", "1 additional hour", "1,000 more people/sq mile",
                   "100 more registrations", "10 pp higher share", "10 pp higher share", "TOD versus FLAT"],
    "Occupancy difference (pp)": hc1.params[predictors] * changes * 100,
    "HC1 95% lower (pp)": hc1.conf_int().loc[predictors, 0] * changes * 100,
    "HC1 95% upper (pp)": hc1.conf_int().loc[predictors, 1] * changes * 100,
    "HC1 p-value": hc1.pvalues[predictors]
})
economic_effects.style.format({"Occupancy difference (pp)": "{:.3f}",
    "HC1 95% lower (pp)": "{:.3f}", "HC1 95% upper (pp)": "{:.3f}", "HC1 p-value": "{:.3e}"})

,Comparison,Occupancy difference (pp),HC1 95% lower (pp),HC1 95% upper (pp),HC1 p-value
price_midpoint,$1 higher proxy,5.563,5.108,6.018,7.931e-127
time_limit_hours,1 additional hour,0.488,-0.127,1.103,1.199e-01
population_density,"1,000 more people/sq mile",0.457,0.411,0.504,8.077e-83
nearby_restaurant_retail_count,100 more registrations,-3.213,-3.689,-2.737,6.502e-40
drive_alone_share,10 pp higher share,-1.981,-2.395,-1.567,6.095e-21
transit_share,10 pp higher share,0.196,-0.690,1.083,6.639e-01
RateType_TOD,TOD versus FLAT,3.300,1.621,4.980,1.177e-04


- **Price:** a $1 higher proxy is associated with **5.56 percentage points** higher occupancy (95% CI: **5.10 to 6.03**). This is a sizeable conditional difference, with the timing and endogeneity qualifications already noted.
- **Time limit:** an additional permitted hour is associated with **0.49 percentage points** higher occupancy, but the interval spans zero (−0.09 to 1.07; p = 0.098).
- **Population density:** an additional **1,000 people per square mile** is associated with **0.46 percentage points** higher occupancy (95% CI: 0.41 to 0.50).
- **Food-service/retail activity:** **100 additional registrations within 0.25 miles** are associated with **3.21 percentage points lower** occupancy (95% CI: −3.66 to −2.76). This conditional association does not mean that opening businesses reduces parking demand.

- **Drive-alone share:** a **10-percentage-point increase** is associated with **1.98 percentage points lower** occupancy (95% CI: −2.39 to −1.57).
- **Transit share:** a **10-percentage-point increase** is associated with **0.20 percentage points higher** occupancy, with substantial uncertainty (95% CI: −0.72 to 1.11; p = 0.675). This is not clear evidence of a nonzero conditional relationship.
- **TOD indicator:** TOD spaces have **3.30 percentage points higher** predicted occupancy than FLAT spaces with the same included covariates (95% CI: 1.87 to 4.73).

The intercept is 0.45634, the fitted occupancy for a FLAT meter when all numeric predictors are zero. That combination is outside the observed support, so the intercept has little standalone economic meaning.

<a id="rubric-5c"></a>

## 5(c) Statistical vs. Economic Significance

**Requirement:** Distinguish statistical significance from economic significance.

**Method:** Use economic units and the uncertainty distinction already in 5(b).

**Evidence:** Original 08 cells 15–16 and original 06 cells 22–23 in 5(b).

**Interpretation:** A precise association does not imply a causal effect; time limit and transit share remain uncertain.

**Status: Complete.** 

<a id="rubric-5d"></a>

## 5(d) Confidence Intervals for Key Parameters

**Requirement:** Calculate and interpret confidence intervals for key parameters.

**Method:** Retain HC1 parameter intervals and percentage-point intervals for all slopes.

**Evidence:** Tables in 5(a)–5(b); price interval interpretation in original 08 cell 15.

**Interpretation:** Price HC1 interval is about 5.11–6.02 occupancy percentage points per dollar; dependence remains unresolved.

**Status: Complete.** 

<a id="rubric-5e"></a>

## 5(e) R² and Model Fit in Economic Context

**Requirement:** Discuss R² and model fit economically.

**Method:** Retain fit/prediction tables and discussion.

**Evidence:** Original 08 cells 17–19.

**Interpretation:** About 27.36% of within-sample variation is explained; substantial variation remains.

**Status: Complete.** 

## 7. Compare fit and prediction

Model 1 uses price alone; Model 2 uses price, population density and nearby food-service/retail count; Model 3 adds time limit, commuting shares and TOD. The corrected middle model has three predictors, as required by the rubric. All comparisons use the same spaces; five-fold errors are verified references from Notebook 06, whose seed-10 splits fit each regression on training rows only.

In [9]:
model1 = sm.OLS(y, sm.add_constant(data[["price_midpoint"]])).fit()
model2 = sm.OLS(y, sm.add_constant(data[["price_midpoint", "population_density",
                                       "nearby_restaurant_retail_count"]])).fit()
models = [model1, model2, model3]
# Verified executed Notebook 06 values; no repeated cross-validation needed here.
cv_rmse = [0.17756368017838295, 0.16820782518065971, 0.16531517414496505]
model_comparison = pd.DataFrame({
    "Model": ["Model 1", "Model 2", "Model 3"], "Predictors": [1, 3, 7],
    "N": [int(m.nobs) for m in models],
    "Price coefficient": [m.params["price_midpoint"] for m in models],
    "R2": [m.rsquared for m in models], "Adjusted R2": [m.rsquared_adj for m in models],
    "AIC": [m.aic for m in models], "BIC": [m.bic for m in models],
    "Mean five-fold RMSE": cv_rmse
})
model_comparison.round(5)

,Model,Predictors,N,Price coefficient,R2,Adjusted R2,AIC,BIC,Mean five-fold RMSE
0,Model 1,1,4016,0.05451,0.15678,0.15657,-2483.67051,-2471.07443,0.17756
1,Model 2,3,4016,0.05999,0.24350,0.24293,-2915.51726,-2890.32509,0.16821
2,Model 3,7,4016,0.05563,0.27362,0.27235,-3070.71241,-3020.32808,0.16532


Model 3 has the highest adjusted R² (**0.27235**) and lowest AIC (**−3,070.71**), BIC (**−3,020.33**) and mean five-fold RMSE (**0.16532**). It explains about **27.36%** of the within-sample variation; held-out error is about **16.53 occupancy percentage points** in root-mean-square terms. Its improvement over Model 2 is modest, about **0.29 percentage points** of RMSE.

These results compare fixed specifications after selection on the available sample. Random folds may share tracts and nearby spaces between training and test sets, so they are not an independent test of the whole selection process or evidence of performance in a new neighborhood.

### Supporting Evidence — Diagnostics and robustness in the final results

All original summary calculations and outputs remain here, even where Section 4 contains the primary checks.

## 8. Diagnostic findings

Notebook 07 contains the residual/fitted, scale-location, residual histogram and Q-Q plots. Uneven residual spread and tract bands support caution about variance and functional form; four fitted occupancies exceed one and are left unclipped. I verify the main test values below rather than treat the favorable normality result as approval of all regression assumptions.

In [10]:
bp = het_breuschpagan(model3.resid, X3)
white = het_white(model3.resid, X3)
reset = linear_reset(model3, power=2, test_type="fitted", use_f=True)
jb = jarque_bera(model3.resid)
tests = pd.DataFrame({
    "Test": ["Breusch–Pagan LM (default Koenker)", "White LM", "RESET F", "Jarque–Bera"],
    "Statistic": [bp[0], white[0], float(reset.fvalue), jb[0]],
    "P-value": [bp[1], white[1], float(reset.pvalue), jb[1]]
})
tests.style.format({"Statistic": "{:.3f}", "P-value": "{:.3e}"})

,Test,Statistic,P-value
0,Breusch–Pagan LM (default Koenker),207.230,3.374e-41
1,White LM,394.973,4.755e-63
2,RESET F,122.078,5.605e-28
3,Jarque–Bera,3.836,1.469e-01


Breusch–Pagan (**p ≈ 3.37 × 10⁻⁴¹**) and White (**p ≈ 4.75 × 10⁻⁶³**) reject constant variance, motivating HC1. RESET (**p ≈ 5.60 × 10⁻²⁸**) strongly questions the mean specification; the conventional test should also be read in light of heteroskedasticity. Jarque–Bera (**p ≈ 0.147**) does not reject normality, which does not resolve either concern.

The 4,016 spaces share only **45 tracts**, and several regressors repeat within each tract. Shared neighborhood shocks can correlate their errors, so treating every space as independent may overstate precision. HC1 and the ordinary row bootstrap do not automatically address that dependence; clustered inference was not demonstrated in the available lectures and is not added here.

## 9. Robustness checks

I summarize the existing Cook’s-distance sensitivity and 1,000-draw row bootstrap from Notebook 07 without repeating its bootstrap. I also verify the simple log-density specification, a stricter 28-active-day sample, and the effect of retaining missing-income spaces. None changes the saved outcome, original data or primary specification.

In [11]:
influence = model3.get_influence()
keep = influence.cooks_distance[0] <= 4 / len(data)
cook_model = sm.OLS(y.loc[keep], X3.loc[keep]).fit(cov_type="HC1")

log_data = data.copy()
assert log_data["population_density"].gt(0).all()
log_data["log_population_density"] = np.log(log_data["population_density"])
log_predictors = ["log_population_density" if v == "population_density" else v for v in predictors]
log_X = sm.add_constant(log_data[log_predictors])
log_ols = sm.OLS(y, log_X).fit()
log_model = sm.OLS(y, log_X).fit(cov_type="HC1")
log_reset = linear_reset(log_ols, power=2, test_type="fitted", use_f=True)

days_data = data.loc[parking.loc[data.index, "active_days"].ge(28)]
days_model = sm.OLS(days_data["occupancy_rate"], sm.add_constant(days_data[predictors])).fit(cov_type="HC1")
income_data = parking[["occupancy_rate", *predictors]].dropna()
income_model = sm.OLS(income_data["occupancy_rate"], sm.add_constant(income_data[predictors])).fit(cov_type="HC1")
fits = [hc1, cook_model, log_model, days_model, income_model]
robustness = pd.DataFrame({
    "Check": ["Primary Model 3", "Cook <= 4/n", "Log population density",
              "At least 28 active days", "Retain missing-income spaces"],
    "N": [int(m.nobs) for m in fits],
    "Price coefficient": [m.params["price_midpoint"] for m in fits],
    "HC1 95% lower": [m.conf_int().loc["price_midpoint", 0] for m in fits],
    "HC1 95% upper": [m.conf_int().loc["price_midpoint", 1] for m in fits],
    "Adjusted R2": [m.rsquared_adj for m in fits]
})
print("Log-density RESET p-value:", float(log_reset.pvalue))
print("Log-density AIC:", log_ols.aic, "BIC:", log_ols.bic)
robustness.round(6)

Log-density RESET p-value: 1.4159409411776104e-19
Log-density AIC: -3075.686296466125 BIC: -3025.3019631751527


,Check,N,Price coefficient,HC1 95% lower,HC1 95% upper,Adjusted R2
0,Primary Model 3,4016,0.055633,0.051081,0.060184,0.272354
1,Cook <= 4/n,3767,0.054769,0.051010,0.058528,0.362017
2,Log population density,4016,0.055160,0.050635,0.059686,0.273255
3,At least 28 active days,1609,0.049628,0.041057,0.058199,0.204005
4,Retain missing-income spaces,4060,0.057162,0.052560,0.061763,0.270828


The price estimate stays positive: **0.05477** after the Cook screen, **0.05516** with log density, **0.04963** in the 1,609-space 28-day sample, and **0.05716** when restoring the 44 missing-income spaces. Log density slightly improves adjusted R² to **0.27325**, but RESET still rejects (**p ≈ 1.42 × 10⁻¹⁹**); it does not repair the original model. Fit differences across restricted samples do not prove improvement.

Notebook 07’s **1,000** row-bootstrap draws (seed **90095**) give a price mean of **0.05548**, standard error **0.002374**, and percentile 95% interval **[0.05091, 0.06011]**. All coefficients are summarized there; these are verified references, not a second resampling exercise. The bootstrap assumes independent rows and holds model selection fixed.

Changing the 24-hour interval cutoff to 12/48 hours or expanding eligibility to 20 active days would require rebuilding **3,707,820** raw transactions and checking predictor linkage. That work and a Sunday-reporting comparison remain uncompleted pending approval; the 28-day check alone is not a completed sensor-outcome sensitivity analysis.

### Conclusion and historical assignment checklist

The original checklist is retained verbatim. The new audit supersedes its completion claims, particularly expected magnitudes, continuous-variable coverage, and Model 3.

## 11. Conclusion

Within this sample, higher stated price proxies are associated with higher monthly occupancy after controlling for the included parking and neighborhood variables. The **5.56-percentage-point** estimate is stable in sign across the completed checks, but it does not support the original negative-price hypothesis as a cross-sectional association. It cannot establish that raising prices increases occupancy.

The seven-variable model predicts somewhat better than the simpler alternatives, but the variance, functional-form and dependence concerns prevent calling it fully validated. The results suggest studying demand-responsive pricing with historically aligned data before making policy recommendations. A parking-availability service would also need time-specific outcomes and genuine future/location validation; monthly occupancy is not an instantaneous probability of finding a space.

## 12. Assignment compliance checklist

This follows `project1-1.pdf` and its 130-point rubric. “Complete” means the requested evidence is present, not that a diagnostic passes or the study establishes causality.

| Rubric / requirement | Status and evidence |
|---|---|
| Economic theory (15): question, 2–3 literature paragraphs, at least two academic sources, hypothesis and expected signs/magnitudes | Complete: Section 1; numerical magnitude is not imposed without a theoretical basis |
| Data/EDA (25): cross-section, 500+ rows, five potential predictors; sources, periods and definitions | Complete: 4,016 estimation spaces; Notebooks 01–04 and Sections 1–2 |
| Data/EDA: mean, median, SD, range, skewness, kurtosis | Complete: Section 2 and Notebook 04 |
| Data/EDA: histogram, boxplot, scatter, correlation heatmap; continuous density/Q-Q and nonlinearities | Complete: Notebook 04, Section 3 and functional-form check in Notebook 07 |
| Data/EDA: outliers, missing patterns, treatment/imputation rationale and sensitivity; Profiling findings | Complete: IQR in Notebook 04, residual screens in Notebook 07, Sections 3 and 9; no imputation needed for final regressors |
| Selection (25): Boruta with parameters, Mallows’ Cp, VIF | Complete: Notebook 05; summarized in Section 4 |
| Selection: one-predictor, 2–3-predictor and broader model | Complete: Models 1/2/3 have 1/3/7 predictors; common sample |
| Selection: wording about statistically significant predictors in Model 3 | Needs instructor clarification: original seven retained as instructed, with time limit and transit share insignificant; not a significance-only subset of all candidates |
| Selection: adjusted R², AIC, BIC, CV and final justification | Complete: Notebook 06 and Sections 4 and 7; best of these three, not every possible model |
| Diagnostics (20): residual/fitted, RESET, BP/White, scale-location, histogram, residual Q-Q/JB | Complete: Notebook 07 and Section 8; variance and RESET concerns remain |
| Diagnostics: Cook’s distance, leverage and DFBETAS | Partial: first two complete; DFBETAS required but no worked lecture example found, so still outstanding |
| Diagnostics: 1,000 bootstrap coefficient estimates and comparison with OLS | Complete: Notebook 07 for every coefficient; price summary in Section 9 |
| Diagnostics: five OR ten folds and out-of-sample prediction | Complete: five training-only folds in Notebook 06; post-selection and shared-tract limits disclosed |
| Results (25): all coefficients, SEs, p-values, economic units, CIs, R² and significance versus size | Complete: Sections 5–7; HC1 clearly distinguished from conventional OLS |
| Limitations (10): bias, policy relevance, external validity and extensions | Complete: Sections 8–11; no causal policy claim |
| Code (10): readable programmatic results, reproducible code and fixed seeds | Executed Notebooks 05–08; Boruta seed 10, CV seed 10, bootstrap seed 90095; expensive results referenced with source notebook |
| Additional project sensor checks | 28-day sample completed; interval-cutoff/20-day/Sunday checks remain, requiring a raw-data rebuild rather than a claim of completion |
| Final group PDF/HTML report with member names and Python source | Not yet packaged/submitted; this notebook and Notebooks 01–07 supply the analysis and source |

The methods used follow the available lectures. Cook’s **4/n** screening rule comes from the project instructions; no new advanced method, clustered estimator or DFBETAS implementation is introduced. The saved raw and processed datasets remain unchanged.

### Literature comparison — audit note

The theory review motivates scarcity and demand-responsive pricing, while the positive price association and negative conditional commercial-count association differ from the simple demand predictions. The original conclusion discusses this tension, but a developed comparison of the final findings with the cited studies remains **Partial** under the PDF’s page-4 assessment criterion. No unverified account of those studies’ numerical results is added.

## Section Summary and Rubric Coverage

5(a)–5(e) have saved tables and interpretation, including the intercept. Conventional narrative intervals and final HC1 intervals remain explicitly distinguished. Page-4 literature comparison is Partial. Final group PDF/HTML packaging is Missing.

Preserved outputs record earlier execution, not a claim that this migrated workflow has been rerun. See [validation evidence](../docs/notebook_validation_summary.md) for fresh execution results and blockers, and the [rubric audit](../docs/rubric_compliance_audit.md) for evidence-backed component status.